# 02 · Decoding with minimum-weight perfect matching

**Goal:** turn the noisy measurement record into a correction, and understand *why decoding is a matching problem*.

Each detector is a node. An error usually flips exactly two detectors, so it becomes an **edge** between two nodes, weighted by how likely that error is. The most likely explanation for the observed detections is the **minimum-weight perfect matching** on this graph — that's what PyMatching computes.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
sys.path.insert(0, os.path.abspath('.'))
import stim
import pymatching
import numpy as np
from surface_code import build_circuit

## From circuit to decoding graph
The decoder's graph comes from the noise model itself — `detector_error_model` reads the error probabilities straight out of the circuit, so edge weights are real log-likelihoods, not hand-tuned.

`decompose_errors=True` splits Y errors into X and Z parts so the problem separates into two independently-matchable graphs. This is the step that makes MWPM applicable at all.

In [2]:
circuit = build_circuit(distance=5, rounds=5, p=0.005)
dem = circuit.detector_error_model(decompose_errors=True)
matching = pymatching.Matching.from_detector_error_model(dem)
print(matching)

<pymatching.Matching object with 120 detectors, 0 boundary nodes, and 502 edges>


## Sample syndromes and decode
`separate_observables=True` gives us the detection events (what the decoder sees) and the true observable flips (the answer key) separately.

In [3]:
sampler = circuit.compile_detector_sampler()
det, obs = sampler.sample(10000, separate_observables=True)
pred = matching.decode_batch(det)

logical_errors = np.sum(pred[:, 0] != obs[:, 0])
print('shots          :', det.shape[0])
print('logical errors :', logical_errors)
print('logical error rate:', logical_errors / det.shape[0])

shots          : 10000
logical errors : 145
logical error rate: 0.0145


## The key idea to be able to say out loud
A *logical error* is not 'an error happened' — errors happen constantly. It's when the decoder's inferred correction differs from the true error by a **logical operator**, so the net effect flips the encoded qubit. Below threshold this is rare and gets rarer with distance.